# 7.6 | Assignment — Build a Production-Style RAG System From Scratch

- Prathyusha Buduru


### Business Use Case

This system acts as an Academic Policy Q&A Assistant.

Users may ask:
- GPA requirements
- Attendance policies
- Academic probation rules

Why retrieval is needed:
- Policies are document-specific
- LLM alone may hallucinate incorrect rules

LLM-only limitations:
- Generates generic answers (e.g., 2.0 GPA instead of actual 2.5)
- Cannot access real policy documents

### Conceptual Understanding

This system uses Retrieval-Augmented Generation (RAG), which combines document retrieval with language generation.

Instead of relying only on a language model, the system:
- Converts text into embeddings to capture semantic meaning
- Retrieves the most relevant document chunks using cosine similarity
- Uses these chunks as context for generating answers

This approach reduces hallucination and ensures answers are grounded in actual documents rather than model assumptions.

In [70]:
import warnings
warnings.filterwarnings("ignore")

from transformers import logging
logging.set_verbosity_error()

In [71]:
!pip install pymupdf sentence-transformers transformers gradio

In [72]:
import fitz
import numpy as np
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
import torch
import pandas as pd

In [73]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [74]:
def clean_text_light(text):
    lines = text.split("\n")
    cleaned = []

    for line in lines:
        line = line.strip()

        # ❌ remove noise patterns
        if line.lower().startswith("question"):
            continue
        if line.lower().startswith("answer"):
            continue

        if line:
            cleaned.append(line)

    return "\n".join(cleaned)

In [75]:
PDF_PATH = "/content/drive/MyDrive/Week_7_Policy_1.pdf"
import fitz

def load_pdf(path):
    doc = fitz.open(path)
    pages = []

    for page_num, page in enumerate(doc):
        text = page.get_text()
        pages.append({
            "page": page_num,
            "text": text
        })

    return pages


pages = load_pdf(PDF_PATH)

# ✅ APPLY CLEANING HERE (THIS WAS MISSING)
pages = [{
    "page": p["page"],
    "text": clean_text_light(p["text"])
} for p in pages]
print("Loaded pages:", len(pages))

Loaded pages: 13


In [76]:
# Analyze text distribution
lengths = [len(p["text"]) for p in pages]

print("Total Pages:", len(pages))
print("Average Length:", sum(lengths)/len(lengths))
print("Max Length:", max(lengths))
print("Min Length:", min(lengths))

Total Pages: 13
Average Length: 1706.4615384615386
Max Length: 2004
Min Length: 1331


### Application & Experimentation

Two chunking strategies were tested:
- Sentence-based chunking
- Paragraph-based chunking

Additionally, system performance was compared between:
- LLM-only responses
- RAG-based responses

Observations:
- LLM-only responses were often generic and sometimes incorrect
- RAG responses were more accurate and grounded in document content
- Paragraph chunking produced better answers due to richer context

In [77]:
import re

def sentence_chunking(pages, chunk_size=3):
    chunks = []
    for p in pages:
        sentences = re.split(r'(?<=[.!?])\s+', p["text"])
        for i in range(0, len(sentences), chunk_size):
            chunk = " ".join(sentences[i:i+chunk_size])
            chunks.append({
                "text": chunk,
                "page": p["page"],
                "type": "sentence"
            })
    return chunks

sent_chunks = sentence_chunking(pages)
print("Sentence chunks:", len(sent_chunks))

Sentence chunks: 65


In [78]:
def paragraph_chunking(pages, min_words=30, max_words=120):
    chunks = []

    for p in pages:
        paragraphs = re.split(r"\n\s*\n", p["text"])

        for para in paragraphs:
            words = para.split()

            if len(words) < min_words:
                continue

            for i in range(0, len(words), max_words):
                sub = " ".join(words[i:i+max_words])

                if len(sub.split()) >= min_words:
                    chunks.append({
                        "text": sub,
                        "page": p["page"],
                        "type": "paragraph"
                    })

    return chunks

In [79]:
para_chunks = paragraph_chunking(pages)
print("Paragraph chunks:", len(para_chunks))

Paragraph chunks: 32


In [80]:
embed_model = SentenceTransformer("all-mpnet-base-v2")

sent_texts = [c["text"] for c in sent_chunks]
para_texts = [c["text"] for c in para_chunks]

sent_embeddings = embed_model.encode(sent_texts, convert_to_numpy=True)
para_embeddings = embed_model.encode(para_texts, convert_to_numpy=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [81]:
def manual_cosine_similarity(query_vector, document_vectors):
    query_vector = np.array(query_vector)
    document_vectors = np.array(document_vectors)

    query_norm = np.linalg.norm(query_vector)
    doc_norms = np.linalg.norm(document_vectors, axis=1)

    return np.dot(document_vectors, query_vector) / (doc_norms * query_norm + 1e-10)

In [82]:
def retrieve(query, embeddings, chunks, top_k=2):

    query_emb = embed_model.encode(query, convert_to_numpy=True)

    scores = manual_cosine_similarity(query_emb, embeddings)

    top_idx = np.argsort(scores)[::-1][:top_k]

    results = []
    for idx in top_idx:
        results.append({
            "text": chunks[idx]["text"]
        })

    return results

In [83]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Fix pad token (important)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model
model = AutoModelForCausalLM.from_pretrained(model_name)

# Fix config (prevents hidden errors)
model.config.pad_token_id = tokenizer.pad_token_id

# Build pipeline
llm = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=-1,              # CPU (safe for Colab)
    max_new_tokens=80,      # keep short for clean answers
    do_sample=False,        # deterministic (important for grading)
    return_full_text=False  # removes prompt repetition
)

print("✅ TinyLlama loaded successfully")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ TinyLlama loaded successfully


In [84]:
def llm_only_answer(query):

    prompt = f"""
Answer the question briefly (1–2 sentences).
Use general academic knowledge only.

Question:
{query}

Answer:
"""

    response = llm(prompt)[0]["generated_text"]

    answer = response.split("Answer:")[-1]

    return answer.strip()

In [97]:
def rag_answer(query, chunk_type="paragraph"):

    # 🔹 Select chunking strategy
    if chunk_type == "paragraph":
        chunks = para_chunks
        embeddings = para_embeddings
    else:
        chunks = sent_chunks
        embeddings = sent_embeddings

    # 🔥 Use ONLY top 1 chunk (THIS IS KEY)
    retrieved = retrieve(query, embeddings, chunks, top_k=2)


    #context = "\n\n".join([r["text"] for r in retrieved])
    context_1 = retrieved[0]["text"]
    context_2 = retrieved[1]["text"] if len(retrieved) > 1 else ""

# Combine for LLM
    context = f"Context 1:\n{context_1}\n\nContext 2:\n{context_2}"

    # 🔥 SIMPLE PROMPT (this is why your other PDF works)
    prompt = f"""
Answer ONLY using the context.
Give a short, direct answer (1–2 sentences).

Context:
{context}

Question:
{query}

Final Answer:
"""

    # 🔹 Generate
    response = llm(prompt)[0]["generated_text"]

    # 🔥 STRONG CLEANING (MOST IMPORTANT PART)
    answer = response.split("Final Answer:")[-1]
    answer = answer.split("Question:")[0]
    answer = answer.split("Context:")[0]

    #return answer.strip(), context
    return answer.strip(), context_1, context_2

In [98]:
def clean_output(text):
    import re
    text = re.sub(r"Question:.*", "", text, flags=re.DOTALL)
    text = re.sub(r"Answer:", "", text)
    return text.strip()

In [99]:
def show_comparison_ui(query):

    llm_ans = llm_only_answer(query)
    rag_ans, context_1, context_2 = rag_answer(query)

    html = f"""
    <h2>Question: {query}</h2>

    <div style="display:flex; gap:20px;">

        <!-- LLM -->
        <div style="width:50%; padding:15px; border:2px solid red; border-radius:10px;">
            <h3>❌ LLM Answer</h3>
            <p>{llm_ans}</p>
        </div>

        <!-- RAG -->
        <div style="width:50%; padding:15px; border:2px solid green; border-radius:10px;">
            <h3>✅ RAG Answer</h3>
            <p>{rag_ans}</p>
        </div>

    </div>

    <!-- CONTEXT DISPLAY -->
    <div style="margin-top:20px;">

        <h3>📄 Retrieved Context</h3>

        <!-- Context 1 -->
        <div style="border:2px solid #4CAF50; padding:12px; margin-bottom:10px; border-radius:8px; background:#f6fff6;">
            <b>Context 1:</b>
            <p>{context_1}</p>
        </div>

        <!-- Context 2 -->
        <div style="border:2px solid #2196F3; padding:12px; border-radius:8px; background:#f4f8ff;">
            <b>Context 2:</b>
            <p>{context_2}</p>
        </div>

    </div>
    """

    display(HTML(html))

In [103]:
from IPython.display import display, HTML

def show_chunking_ui(query):



    #para_ans, _ = rag_answer(query, "paragraph")
    #sent_ans, _ = rag_answer(query, "sentence")

    para_ans, _, _ = rag_answer(query, "paragraph")
    sent_ans, _, _ = rag_answer(query, "sentence")

    para_ans = clean_output(para_ans)
    sent_ans = clean_output(sent_ans)

    html = f"""
    <div style="font-family: Arial; margin-bottom:30px;">

        <h3 style="margin-bottom:10px;">
            Question: {query}
        </h3>

        <div style="display:flex; gap:20px;">

            <!-- PARAGRAPH BOX -->
            <div style="
                flex:1;
                border:2px solid #4A90E2;
                border-radius:10px;
                padding:15px;
                background:#eef5ff;
            ">
                <h4 style="color:#4A90E2;">📘 Paragraph Chunking</h4>
                <p>{para_ans}</p>
            </div>

            <!-- SENTENCE BOX -->
            <div style="
                flex:1;
                border:2px solid #f5a623;
                border-radius:10px;
                padding:15px;
                background:#fff7e6;
            ">
                <h4 style="color:#f5a623;">📙 Sentence Chunking</h4>
                <p>{sent_ans}</p>
            </div>

        </div>
    </div>
    """

    display(HTML(html))

In [101]:
queries = [
    "What GPA is required to remain in good academic standing?",
    "What happens if attendance falls below 75%?",
    "What is academic probation?"
]

for q in queries:
    show_comparison_ui(q)

### Retrieval Performance Analysis

- RAG retrieves relevant policy-specific chunks before answering
- LLM-only generates generic responses (hallucination observed)
- RAG answers are grounded in actual documents

Therefore, RAG significantly improves accuracy and reliability.

In [104]:
queries = [
    "What GPA is required?",
    "What happens if attendance falls below 75%?"
]

for q in queries:
    show_chunking_ui(q)

### Chunking Strategy Comparison

Sentence Chunking:
- Breaks text into smaller units
- May lose full context across sentences
- Useful for precise retrieval but can miss complete meaning

Paragraph Chunking:
- Preserves semantic context
- Provides more complete information
- Better for answering descriptive questions

Conclusion:
Paragraph chunking performs better for this use case because it retains more contextual information, leading to more accurate and complete answers.

### Business Impact Analysis

RAG improves answer reliability by grounding responses in actual documents rather than relying on model memory.

Compared to LLM-only systems:
- LLM generates generic and sometimes incorrect answers (hallucination)
- RAG retrieves relevant company-specific policies before answering

This significantly reduces hallucination and increases accuracy.

RAG also improves domain specificity:
- Answers are tailored to the provided documents
- Responses reflect real organizational rules instead of general knowledge

Failure Cases:
- If retrieval selects irrelevant chunks, answers may still be incorrect
- If important context is split across chunks, the model may miss key details
- Ambiguous queries can lead to partially correct answers



Unlike traditional machine learning assignments, this project focuses on a Retrieval-Augmented Generation (RAG) system rather than training predictive models.

EDA, model training, and bias-variance tradeoff are not directly applicable because:
- The system uses pre-trained embedding models and LLMs
- No supervised training or labeled dataset is involved

Instead, this project evaluates:
- Retrieval accuracy (relevant chunk selection)
- Answer grounding (context-based responses)
- Hallucination reduction (RAG vs LLM-only comparison)

The comparison between LLM-only and RAG-based responses serves as the primary evaluation framework.